We consider a binomial likelihood. That is, $x\mid \theta \sim \text{Bin}(100, \theta)$, where $\theta \in (0, 1)$. In this case, the true score is
$$
\nabla_\theta \log p(x \mid \theta) = \frac{x}{\theta} - \frac{100-x}{1-\theta}
$$

We will choose $p(\theta) = \text{Beta}(\alpha, \beta)$ as the proposal distribution for $\theta$ to train the score neural network. And we will show that the accuraccy of the score estimation error depends on the density.

In [ ]:
import os    
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import math
import matplotlib.pyplot as plt
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def set_seed(seed):
    np.random.seed(seed)  # NumPy RNG
    torch.manual_seed(seed)  # PyTorch CPU RNG
    torch.cuda.manual_seed(seed)  # Current GPU RNG
    torch.cuda.manual_seed_all(seed)  # All GPUs (if using DataParallel or multi-GPU)

    torch.backends.cudnn.deterministic = True  # Use deterministic algorithms
    torch.backends.cudnn.benchmark = False     # Disable autotuner for reproducibility

In [ ]:
set_seed(3)

## Generate training data

In [ ]:
alpha = 5
beta = 5
n_bin = 100 

In [ ]:
sample_size = 10000
theta_r0 = np.random.beta(alpha, beta, size = sample_size)
x_r0 = np.random.binomial(n = n_bin, p = theta_r0, size = sample_size)

val_theta_r0 = np.random.beta(alpha, beta, size = sample_size)
val_x_r0 = np.random.binomial(n = n_bin, p = val_theta_r0, size = sample_size)

theta_r0 = torch.tensor(theta_r0, dtype = torch.float32).view(-1, 1)
x_r0 = torch.tensor(x_r0, dtype = torch.float32).view(-1, 1)
val_theta_r0 = torch.tensor(val_theta_r0, dtype = torch.float32).view(-1, 1)
val_x_r0 = torch.tensor(val_x_r0, dtype = torch.float32).view(-1, 1)

## Neural Network Function

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class ELU_LikeScoreMatchingNN(nn.Module):
    def __init__(self, theta_dim, x_dim, obs_size, hidden_size, num_layers = 1):
        super(ELU_LikeScoreMatchingNN, self).__init__()
        
        layers = [nn.Linear(theta_dim + x_dim, hidden_size), nn.ELU()]

        # Add hidden layers based on num_layers
        for _ in range(num_layers - 1):
            layers.append(nn.Linear(hidden_size, hidden_size))
            layers.append(nn.ELU())

        # Output layer to match the desired output size
        layers.append(nn.Linear(hidden_size, theta_dim))

        # Combine all layers into a sequential model
        self.layers = nn.Sequential(*layers)
        self.obs_size = obs_size

    def forward(self, theta, x):
        if len(theta.shape) == 1: # if one-dimensional
            theta = theta.view(-1, 1)

        if len(x.shape) == 1:
            x = x.view(-1, 1)
        x_dim = int(x.shape[1] / self.obs_size)
        score = self.layers( torch.cat( (theta.repeat_interleave(self.obs_size, dim = 0), x.reshape(-1, x_dim)), dim = 1 ) ).view(theta.shape[0], self.obs_size, theta.shape[1]).sum(dim = 1)
        return score


import matplotlib.pyplot as plt
import time

# we will add a weight function g() to meet the boundary condition
def Like_score_loss(model, theta, x, prop_score, g, g1):
    # g: weight function, takes input (theta, x), output dimension is the same as theta
    # g1: first derivative of g (the diagonal part, \partial g(\theta, x)_j / \partial \theta_j), output dimension is the same as theta
    # we require g and g1 to be able to address matrix input and do element-wise mapping
    
    theta.requires_grad_(True)
    score = model(theta, x)
    loss1 = torch.norm(score * g(theta, x)**(1/2), dim = -1) ** 2 / 2.
    loss3 = ((score * g(theta, x)) * prop_score).sum(dim = -1)
    
    loss2 = torch.zeros(theta.shape[0]).to(device)
    for i in range(theta.shape[1]):
        grad2 = torch.autograd.grad(outputs = score[:, i].sum(), inputs = theta, create_graph=True)[0][:, i]
        loss2 += grad2 * (g(theta, x)[:, i]) + score[:, i] * g1(theta, x)[:, i]
    loss = loss1 + loss2 + loss3
    return loss.mean()

# The training function, where the dataloader contains (theta, x, prop_score)
def train(model, optimizer, dataloader, val_dataloader, g, g1, num_epochs = 101):
    model.to(device)

    # record training loss and validation loss at each epoch and then plot
    train_losses = []
    val_losses = []
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train() 
        total_loss = 0.0
        for batch_sample in dataloader:
            optimizer.zero_grad()
            batch_theta, batch_x, batch_prop_score = batch_sample
            batch_theta, batch_x, batch_prop_score = batch_theta.to(device), batch_x.to(device), batch_prop_score.to(device)
            loss = Like_score_loss(model, batch_theta, batch_x, batch_prop_score, g, g1)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()    

        model.eval()
        total_loss_val = 0.0
        # with torch.no_grad():
        for val_batch_sample in val_dataloader:
            val_batch_theta, val_batch_x, val_batch_prop_score = val_batch_sample
            val_batch_theta, val_batch_x, val_batch_prop_score = val_batch_theta.to(device), val_batch_x.to(device), val_batch_prop_score.to(device)
            val_loss = Like_score_loss(model, val_batch_theta, val_batch_x, val_batch_prop_score, g, g1)
            total_loss_val += val_loss.item()    
        if epoch % 1 == 0:
            print(f'Epoch {epoch+1}/{num_epochs}, Training Loss: {round(total_loss / len(dataloader), 3)}, Validation Loss: {round(total_loss_val / len(val_dataloader), 3)}')
        
        train_losses.append(total_loss / len(dataloader))
        val_losses.append(total_loss_val / len(val_dataloader))
    
    end_time = time.time()
    total_duration = end_time - start_time
    print(f'Total training time: {round(total_duration, 2)} seconds')


# No weight function (weight = 1)
def g(theta, x):
    return 1.0 * torch.ones(theta.shape[0]).view(-1, 1).to(device)

def g1(theta, x):
    return 0.0 * torch.ones(theta.shape[0]).view(-1, 1).to(device)

## Train the NN

In [ ]:
prop_score_r0 = (alpha - 1) / theta_r0 - (beta - 1) / (1 - theta_r0)
val_prop_score_r0 = (alpha - 1) / val_theta_r0 - (beta - 1) / (1 - val_theta_r0)

In [ ]:
## Training
# Create DataLoader
batch_size = 500 # 500 
dataset = TensorDataset(theta_r0, x_r0, prop_score_r0)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)
val_dataset = TensorDataset(val_theta_r0, val_x_r0, val_prop_score_r0)
val_dataloader = DataLoader(val_dataset, batch_size = batch_size, shuffle = False)

# Create model and optimizer
theta_dim = 1
x_dim = 1
obs_size = 1
hidden_size = 32 # 8 # 16
model = ELU_LikeScoreMatchingNN(theta_dim, x_dim, obs_size, hidden_size, num_layers = 3)

learning_rate = 1e-4 # 1e-4
optimizer = optim.Adam(model.parameters(), lr = learning_rate, weight_decay = 1e-5) 

# train the model
train(model, optimizer, dataloader, val_dataloader, g, g1, num_epochs = 400) 

# Error at Low Density Area

In [ ]:
def cal_error(theta_star):
    sample_size = 100000
    x = np.random.binomial(n = n_bin, p = theta_star, size = sample_size).reshape(-1, 1) # draw x from theta_star
    theta = np.random.beta(x + alpha, n_bin - x + beta).reshape(-1, 1) # draw theta from the posterior given x
    x, theta = torch.tensor(x, dtype = torch.float32), torch.tensor(theta, dtype = torch.float32)
    score_est = model.cpu()(theta, x).ravel()
    score_true = ( x/theta- (n_bin - x)/(1 - theta) ).ravel()
    return ( (score_est - score_true)**2 ).mean().item()

In [ ]:
theta_star_set = np.arange(0.01, 1, 0.01)
error_set = np.array([cal_error(theta_star) for theta_star in theta_star_set])

In [ ]:
import scipy.stats

# Create the plot
fig, ax1 = plt.subplots()

# Plot the first data set on the primary y-axis (left side)
line1, = ax1.plot(theta_star_set, error_set, label="Error")
ax1.set_xlabel(r"$\theta_0$", fontsize=20)
ax1.set_ylabel(r"$\text{Error}\,(\theta_0)$", fontsize=20)
ax1.tick_params(axis='both', labelsize=20) 

ax1.grid(True)
ax1.set_yscale("log")

# Create a twin y-axis for the second plot (right side)
density_set = scipy.stats.beta.pdf(theta_star_set, alpha, beta)
ax2 = ax1.twinx()
line2, = ax2.plot(theta_star_set, density_set, color="orange", label=r"Density")
ax2.set_ylabel(r"$\pi\,(\theta_0)$", fontsize=20)
ax2.tick_params(axis='both', labelsize=20) 

# Combine both legends and place them at the bottom
lines = [line1, line2]
labels = [line.get_label() for line in lines]
fig.legend(lines, labels, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.2), fontsize=20)

# plt.savefig("binomial.pdf", format="pdf", dpi=300, bbox_inches="tight")
plt.show()